In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
print(hf_token[:10])

hf_EditoLN


In [3]:
!pip install -q transformers accelerate

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it", token=hf_token)

model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    token=hf_token
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded successfully!


In [5]:
input_text = "Patient has high fever and low oxygen. What could be the issue?"
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

inputs = tokenizer(input_text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0]))

<bos>Patient has high fever and low oxygen. What could be the issue?

**Answer:**

**Possible causes of high fever and low oxygen include:**

* **Sepsis:** A life-threatening condition where the body's immune system overreacts to an infection, causing inflammation and tissue damage.
* **Pyelonephritis:** Inflammation of the kidneys, which can be caused by bacteria or viruses.
* **Meningitis:** Inflammation of the membranes surrounding the brain and spinal cord, which can be caused by viruses or bacteria.
* **Cardiopulmonary


In [6]:
patient_data = { "heart_rate": 110,
                 "spo2": 88,
                 "temperature": 101
               }

In [7]:
alerts=[]

if patient_data["heart_rate"]>100:
    alerts.append("High heart rate")

if patient_data["spo2"]<92:
    alerts.append("Low oxygen level")

if patient_data["temperature"]>100:
    alerts.append("Fever")

print(alerts)

['High heart rate', 'Low oxygen level', 'Fever']


In [8]:
input_text = f"""
Patient vitals:
Heart Rate: {patient_data['heart_rate']}
SpO2: {patient_data['spo2']}
Temperature: {patient_data['temperature']}

Detected issues: {', '.join(alerts)}

Explain possible medical condition and advice.
"""

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=150)

print(tokenizer.decode(outputs[0]))

<bos>
Patient vitals:
Heart Rate: 110
SpO2: 88
Temperature: 101

Detected issues: High heart rate, Low oxygen level, Fever

Explain possible medical condition and advice.
**Possible medical condition:**

- **Cardiogenic shock:** A sudden drop in blood pressure and heart rate that can occur when the body is deprived of oxygen.

**Advice:**

- **Immediate medical attention:** Call 911 immediately.
- **Oxygen therapy:** If available, provide oxygen to help improve oxygenation.
- **Cardiogenic resuscitation:** If possible, start CPR to help maintain blood flow and oxygenation.
- **Monitor vital signs:** Monitor the patient's heart rate, oxygen saturation, and temperature closely.
- **Evaluate underlying causes:** Determine the underlying cause of the shock, such as a heart attack, sepsis, or trauma.
- **Provide supportive care:** Ensure the patient is comfortable and receiving


In [9]:
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("AI Medical Analysis\n")
print(response)

AI Medical Analysis


Patient vitals:
Heart Rate: 110
SpO2: 88
Temperature: 101

Detected issues: High heart rate, Low oxygen level, Fever

Explain possible medical condition and advice.
**Possible medical condition:**

- **Cardiogenic shock:** A sudden drop in blood pressure and heart rate that can occur when the body is deprived of oxygen.

**Advice:**

- **Immediate medical attention:** Call 911 immediately.
- **Oxygen therapy:** If available, provide oxygen to help improve oxygenation.
- **Cardiogenic resuscitation:** If possible, start CPR to help maintain blood flow and oxygenation.
- **Monitor vital signs:** Monitor the patient's heart rate, oxygen saturation, and temperature closely.
- **Evaluate underlying causes:** Determine the underlying cause of the shock, such as a heart attack, sepsis, or trauma.
- **Provide supportive care:** Ensure the patient is comfortable and receiving


In [10]:
input_text = f"""
You are a medical assistant AI.

Patient vitals:
- Heart Rate: {patient_data['heart_rate']}
- SpO2: {patient_data['spo2']}
- Temperature: {patient_data['temperature']}

Detected issues: {','.join(alerts)}

Give:
1. Possible condition
2. Risk level (Low/Medium/High)
3. Simple advice
"""

In [11]:
patients = [
    {"heart_rate": 110, "spo2": 88, "temperature": 101},
    {"heart_rate": 75, "spo2": 95, "temperature": 98},
]

for i, patient_data in enumerate(patients):
    print(f"\n--- Patient {i+1} ---")

    alerts = []

    if patient_data["heart_rate"]>100:
        alerts.append("High heart rate")
    if patient_data["spo2"]<92:
        alerts.append("Low Oxygen")
    if patient_data["temperature"]>100:
        alerts.append("Fever")

input_text = f"""
Patient vitals:
{patient_data}
Issues: {','.join(alerts)}
Explain condition and advice.
"""

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


--- Patient 1 ---

--- Patient 2 ---

Patient vitals:
{'heart_rate': 75, 'spo2': 95, 'temperature': 98}
Issues: 
Explain condition and advice.
The patient's vitals show a heart rate of 75, a spo2 of 95, and a temperature of 98. These values are considered normal.

**Condition:** The patient is in a state of homeostasis, meaning their body is maintaining a stable internal environment despite external changes.

**Advice:** The patient should continue to monitor their vitals and seek medical attention if they experience any changes or if their condition worsens.


In [12]:
!pip install -q gradio

In [13]:
medical_knowledge = [
    "High heart rate may indicate stress, fever or cardiovascular issues.",
    "Low SpO2 levels can indicate respiratory problems or oxygen deficiency.",
    "Fever is usually a sign of infection or inflammation.",
    "Normal SpO2 levels are typically between 95% and 100%",
    "Heart rate above 100 bpm is considered tachycardia."
]

In [14]:
def retrieve_info(alerts):
    relevant_info = []

    for alert in alerts:
        for info in medical_knowledge:
            if alert.lower().split()[0] in info.lower():
                relevant_info.append(info)

    return "\n".join(relevant_info)

In [15]:
import gradio as gr

retrieved_info = retrieve_info(alerts)

def calculate_risk_score(heart_rate, spo2, temperature):
    score = 0

    if heart_rate > 100:
        score += 2
    if spo2 < 92:
        score += 3
    if temperature > 100:
        score += 2

    return score

def get_risk_level(score):
    if score <= 2:
        return "Low"
    elif score <= 4:
        return "Medium"
    else:
        return "High"

def analyze_patient(heart_rate, spo2, temperature):
    alerts = []

    if heart_rate > 100:
        alerts.append("High heart rate")
    if spo2 < 92:
        alerts.append("Low oxygen")
    if temperature > 100:
        alerts.append("Fever")

    score = calculate_risk_score(heart_rate, spo2, temperature)
    risk_level = get_risk_level(score)

    input_text = f"""
    You are a professional medical assistant AI.
    
    Patient vitals:
    -Heart Rate: {heart_rate} bpm
    -SpO2: {spo2} %
    -Temperature: {temperature} °F

    Detected Issues: {', '.join(alerts) if alerts else 'None'}

    Risk Score: {score}
    Predicted Risk Level: {risk_level}

    Relevant medical knowledge:
    {retrieved_info if retrieved_info else 'None'}

    Instructions:
    1. Clearly identify the possible condition
    2. Mention the risk score and risk level
    3. Provide short, practical medical advice
    4. Keep response structured and easy to read

    Use the above knowledge to provide:

    Output Format:
    Condition:
    Risk score:
    Risk level:
    Advice:
    """

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=80)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Condition:" in response:
        clean_response = "Condition:" + response.split("Condition:")[-1].strip()
    else:
        clean_response = "Condition: Not clearly identified\n" + response.strip()

    if not clean_response.strip():
        clean_response = "AI could not generate response. Please try again."

    return f"""
    PATIENT ANALYSIS REPORT
    -----------------------

    Risk Score: {score}
    {risk_color} Risk Level: {risk_level}

    Detected Issues:
    {','.join(alerts) if alerts else 'None'}

    Medical Insights:
    {retrieved_info if retrieved_info else 'No relevant info found'}

    AI Diagnosis & Advice:
    {clean_response if clean_response.strip() else 'AI response unavailable. Please try again!'}
    """

In [16]:
import gradio as gr

def analyze_patient(heart_rate, spo2, temperature):
    alerts = []

    if heart_rate > 100:
        alerts.append("High heart rate")
    if spo2 < 92:
        alerts.append("Low oxygen")
    if temperature > 100:
        alerts.append("Fever")

    score = calculate_risk_score(heart_rate, spo2, temperature)
    risk_level = get_risk_level(score)
    if risk_level == "Low":
        risk_color = "Green"
    elif risk_level == "Medium":
        risk_color = "Yellow"
    else:
        risk_colour = "Red"
    
    retrieved_info = retrieve_info(alerts)

    input_text = f"""
    Patient vitals:
    -Heart Rate: {heart_rate}
    -SpO2: {spo2}
    -Temperature: {temperature}

    -Risk Score: {score}
    -Risk Level: {risk_level}

    {retrieved_info}

    Use the above knowledge to provide:
    -Condition:
    -Advice:
    Always prioritize patient safety and suggest seeking medical help if risk is high.
    """

    inputs = tokenizer(input_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response   # keep simple for now


with gr.Blocks() as demo:
    
    gr.Markdown("#AI Healthcare Risk Detection System")
    gr.Markdown("### Enter patient vitals to analyze health risk and get AI-powered insights")

    with gr.Row():
        heart_rate = gr.Slider(40, 150, label="Heart Rate (bpm)")
        spo2 = gr.Slider(70, 100, label="SpO2 (%)")
        temperature = gr.Slider(95, 105, label="Temperature (°F)")

    submit_btn = gr.Button("Analyze Patient")
    clear_btn = gr.Button("Clear")
    score_display = gr.Markdown()

    output = gr.Markdown(label="Medical Report")

    submit_btn.click(
        fn=analyze_patient,
        inputs=[heart_rate, spo2, temperature],
        outputs=[score_display, output],
        show_progress=True
    )
    clear_btn.click(
        fn=lambda: ("", "", "", ""),
        inputs=[],
        outputs=[heart_rate, spo2, temperature, output]
    )
    gr.Examples(
        examples=[
            [120,88,102],
            [80,97,98],
            [110,90,101]
        ],
        inputs=[heart_rate, spo2, temperature]
    )
    gr.Markdown("Disclaimer: This AI tool is for educational purposes only and not a substitute for professional medical advice.")

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://961ca9fad07f2ef04e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
